# Data Drift Detection with SAP HANA APL

**Automated Predictive Library (APL) — Step-by-Step Tutorial**

---

**APL (Automated Predictive Library)** is a component of SAP HANA that provides automated analytics,
including the ability to detect **data drift** — changes in the statistical properties of data over time.

The drift detector helps you spot statistical changes or deviations between a **test dataset** and a
**reference**. The reference can be a past snapshot of the data, a specific population segment, or a
theoretical distribution such as Benford's law. Typical use cases include:

- **ML model monitoring** — compare a scoring or inference dataset against the training baseline; if feature
  distributions shift after training, the model's predictions become less reliable and you can retrain
  before accuracy degrades
- **Survey tracking** — compare this year's employee survey results against last year's, broken down by country
- **Population comparison** — compare demographic or behavioral profiles across workforce segments
  (e.g. male vs. female staff)
- **Fraud detection** — compare payment amounts by legal entity against Benford's law to surface
  anomalous patterns

**What you will learn:**

- How to connect to SAP HANA from Python using a `ConnectionContext`
- What data drift is and how APL measures it with the **deviation indicator**
- How to compare two populations in a single call with `fit_detect()`
- How to control detection sensitivity with the `threshold` parameter
- How to explore drift results with an interactive HTML report (inline or as a shareable file)
- How to access drift details programmatically — per-variable indicators, category frequencies, and more
- How to use **supervised drift detection** when a target label is available
- How to detect drift **independently per segment** with `segment_column_name`
- How to persist a fitted drift model to SAP HANA and reload it in a later session with `ModelStorage`

## 1. Setup

We start by importing the required libraries and configuring the APL logger.

The APL logger controls how much internal detail APL prints during execution. Setting it to
`logging.WARNING` keeps the output clean — only warnings
and errors will be shown. You can change it to `logging.INFO` for a full execution trace,
or to `logging.ERROR` to suppress warnings.

In [ ]:
import logging

import hana_ml
from hana_ml import dataframe as hd
from hana_ml.algorithms.apl.drift_detector import DriftDetector
from hana_ml.algorithms.apl import apl_base
from hana_ml.model_storage import ModelStorage

# Keep APL log output clean — change to logging.INFO for verbose output
apl_base.config_logger(
    log_level=logging.WARNING,
    #log_path='C:/My_folder',
    #logfile_name='HANA_PYTHON_APL_TRACE',
)

print(f"hana_ml version: {hana_ml.__version__}")

## 2. Connect to SAP HANA

Open a `ConnectionContext` — the object that manages the database session. Two authentication
options are available:

- **Explicit credentials** — provide host, port, user, and password directly in the notebook.
- **User key** — reference a key stored in the SAP HANA secure user store (`hdbuserstore`).
  This is the recommended approach for shared or production notebooks because credentials are
  never written in plain text.

Everything that follows runs **inside SAP HANA**. The Python client only sends instructions and
receives results; the data never leaves the database unless you explicitly pull it.

Two types of DataFrames are used throughout this notebook:

- **HANA DataFrame** (`hana_ml.DataFrame`) — a lazy reference to a SQL query or table in SAP HANA.
  No data is transferred to the Python client; operations on it are translated into SQL and
  executed on the database.
- **pandas DataFrame** — an in-memory table on the Python client. You obtain one by calling
  `.collect()` on a HANA DataFrame, which executes the underlying query and pulls the result
  set to the client.

In [ ]:
# Option 1 — explicit credentials
HDB_HOST =
HDB_PORT =
HDB_USER =
HDB_PASS =

conn = hd.ConnectionContext(
    HDB_HOST, HDB_PORT, HDB_USER, HDB_PASS, encrypt=True, sslValidateCertificate=False
)

# Option 2 — user key stored in hdbuserstore (recommended for shared/production notebooks)
# conn = hd.ConnectionContext(userkey='mykey', encrypt=True, sslValidateCertificate=False)

print("Connected to SAP HANA.")

## 3. The Dataset

This notebook uses the **US Census (Adult) dataset** from the `APL_SAMPLES` schema, which is
shipped with SAP HANA APL — no file upload is needed.

The table records demographic and employment information for individuals: age, education,
occupation, relationship status, workclass, and more.

**Scenario:** we compare the demographic profiles of **male** and **female** respondents to
measure how much the two populations differ across these variables. This is a classic use case
for drift detection:

- The **reference dataset** is the population that was used to train an ML model (here, male respondents).
- The **new dataset** is a later batch of scoring data whose distribution we want to verify
  (here, female respondents).

If the new population's feature distributions differ significantly from the reference, any model
trained on the reference may produce unreliable predictions on the new data.

In [ ]:
hana_census = conn.table("CENSUS", "APL_SAMPLES")

print(f"Total rows : {hana_census.count()}")
print(f"Columns    : {hana_census.columns}")
hana_census.head(5).collect()

In [ ]:
# Exclude the split criterion (sex) and the label reserved for supervised detection (class)
EXCLUDE = {"sex", "class"}
FEATURES = [c for c in hana_census.columns if c not in EXCLUDE]

# Reference population: male respondents
hana_ref = hana_census.filter("\"sex\" = 'Male'").select(FEATURES)

# New population: female respondents (simulates a later scoring batch)
hana_new = hana_census.filter("\"sex\" = 'Female'").select(FEATURES)

print(f"Reference (Male)  : {hana_ref.count()} rows")
print(f"New       (Female): {hana_new.count()} rows")
print(f"Features          : {FEATURES}")

## 4. Detect Drift in One Call: `fit_detect()`

`fit_detect()` is the all-in-one method for drift detection. It:

1. Fits an internal statistical model on the **reference dataset**
2. Applies it to the **new dataset**
3. Returns a `DataFrame` with a **deviation indicator** for each variable

The whole operation runs inside SAP HANA — no data is pulled to the Python client.

> **Note:** `fit_detect()` does not retain the
> internal model binary. If you need to save the fitted model and reuse it in a later
> session (e.g. to monitor incoming batches without refitting), use `fit()` + `detect()`
> instead. See **Section 9 — Two-Phase Workflow** for details.

In [ ]:
detector = DriftDetector()

results = detector.fit_detect(
    reference_data=hana_ref,
    new_data=hana_new,
    build_report=True,
)

print("Drift detection complete.")
results.collect()

The result table has one row per variable:

| Column | Description |
|---|---|
| `Variable` | Name of the feature column. |
| `Deviation Indicator` | Drift measure for this variable. Ranges from `0.0` (identical distributions) to `1.0` (maximally different distributions). Rows are sorted in descending order so the most drifted variables appear first. APL uses a chi-squared test to compute the deviation indicator. |

**The `threshold` parameter** controls which variables are flagged as drifted in the returned
DataFrame. Only variables whose deviation indicator exceeds the threshold are included in the
result. The default is `0.95`. Lowering it (e.g. to `0.5`) widens the net and includes
variables with moderate drift; raising it (e.g. to `0.99`) restricts results to the most
severely drifted variables.

## 5. The Drift Report

Passing `build_report=True` to `fit_detect()` (or `detect()`) tells APL to prepare all
the debrief data needed for the interactive report in the same pass — no extra database
round-trip required afterward.

- `generate_notebook_iframe_report()` — renders the report **inline** in this notebook
- `generate_html_report(name)` — saves it as a **self-contained HTML file** (all charts
  and data included, no Python or SAP HANA required to view it) that you can share with
  stakeholders

The report has multiple tabs. The tabs that appear depend on the data:

| Tab | What you see |
|---|---|
| **Overview** | Detection summary: reference and new dataset sizes, build date, target variable (if any), threshold. An alert banner shows whether drift was detected. |
| **Variable Drift** | Bar chart of the top 20 variables ranked by deviation indicator. Only shown when at least one variable exceeds the threshold. |
| **Category Drift** | Per-variable bar charts showing deviation indicators at the category level. Nominal variables appear as-is; numeric variables are automatically discretized into 20 intervals. |
| **Category Frequencies** | Per-variable side-by-side bar charts comparing the percentage weight of each category in the reference vs. the new dataset. Shows the top 20 categories by absolute percentage change. |

In [ ]:
# Render the report inline in this notebook
detector.generate_notebook_iframe_report()

# Save as a standalone HTML file you can share with stakeholders
detector.generate_html_report("census_drift_report")
print("Report saved to census_drift_report.html")

## 6. Programmatic Access

The interactive report summarizes the drift analysis visually. This section shows how to
access the same information programmatically — useful when you need to extract specific
numbers, log results, or feed them into a monitoring pipeline.

`get_debrief_report()` exposes the underlying statistical tables that APL computes during
detection. Each report returns a HANA DataFrame that you can collect and analyze.
The full list of available reports and their contents is documented in the
[APL Deviation Reports reference](https://help.sap.com/docs/apl/7223667230cb471ea916200712a9c682/86d1768c5b4c433d80a5e21bb96bb639.html).

### Drift by Variable

`Deviation_ByVariable` returns the deviation indicator for each variable. Passing
`deviation_threshold` filters the result to variables that exceed the given threshold,
mirroring the behavior of the `threshold` parameter in `fit_detect()` and `detect()`.

In [ ]:
by_variable = (
    detector.get_debrief_report("Deviation_ByVariable", deviation_threshold=0.5)
    .deselect("Oid")
    .collect()
)
print("Variables with deviation indicator > 0.5:")
by_variable

### Category Frequencies

`Deviation_CategoryFrequencies` shows how the weight of each category shifted between the
reference and new datasets. Filtering on a single variable shows which specific categories drove the drift.

In [ ]:
cat_freq = (
    detector.get_debrief_report("Deviation_CategoryFrequencies")
    .deselect("Oid")
    .collect()
)

print("Top 10 category shifts for 'occupation':")
(
    cat_freq[cat_freq["Variable"] == "occupation"]
    .sort_values("Abs % Change", ascending=False)
    .head(10)
    .reset_index(drop=True)
)

### Category-Level Deviation Indicators

`Deviation_ByCategory` provides deviation indicators at the **category level** — one row per
variable-category pair — rather than the variable-level summary above.

In [ ]:
by_category = (
    detector.get_debrief_report("Deviation_ByCategory")
    .deselect("Oid")
    .collect()
    .sort_values(["Variable", "Deviation Indicator"], ascending=[True, False])
)
print("Category-level drift for 'relationship':")
by_category[by_category["Variable"] == "relationship"].reset_index(drop=True)

## 7. Supervised Drift Detection

By default, `DriftDetector` operates in **unsupervised mode**: it examines each variable's
marginal distribution independently. This can miss drift that only manifests in the
**joint distribution** between variables.

A classic illustration: compare a standard card deck (26 black clubs, 26 black spades,
26 red diamonds, 26 red hearts) against a non-standard one (26 red clubs, 26 red spades,
26 black diamonds, 26 black hearts). Neither `Suit` nor `Color` deviates individually —
the marginal distribution of each variable is identical in both decks. Yet the relationship
between the two variables has completely flipped.

Passing a **target label** via the `label` parameter activates **supervised mode**. APL
then detects drift in the *joint distribution* of each feature with the target variable,
surfacing shifts that are invisible when variables are examined in isolation.

> **Adapt the label column name** to match your dataset. Here we use the `class` column
> (income category: `>50K` or `<=50K`).

In [ ]:
LABEL_COLUMN = "class"  # income category: >50K / <=50K

hana_ref_sup = hana_census.filter("\"sex\" = 'Male'").select(FEATURES + [LABEL_COLUMN])
hana_new_sup = hana_census.filter("\"sex\" = 'Female'").select(
    FEATURES + [LABEL_COLUMN]
)

detector_sup = DriftDetector()

results_sup = detector_sup.fit_detect(
    reference_data=hana_ref_sup,
    new_data=hana_new_sup,
    label=LABEL_COLUMN,
    build_report=True,
)

print("Supervised drift detection complete.")
results_sup.collect()

In [ ]:
detector_sup.generate_notebook_iframe_report()

detector_sup.generate_html_report("census_drift_supervised_report")
print("Report saved to census_drift_supervised_report.html")

Compared with the unsupervised report, the supervised report has four additional tabs:

| Tab | What you see |
|---|---|
| **Target-Based Category Drift** | Deviation indicators at the category level, measuring drift in the *joint distribution* of each feature category with the target. A category whose marginal frequency is unchanged but whose target distribution shifted will be flagged here even though the plain **Category Drift** tab shows no deviation. |
| **Group Drift** | Deviation indicators using **supervised grouping** learned on the reference data: for numeric variables, intervals whose boundaries are determined by the relationship with the target; for nominal variables, categories are merged into groups based on their target distribution similarity. Measures marginal distribution drift of these groups. |
| **Group Frequencies** | Side-by-side comparison of reference vs. new percentage weight per supervised group. |
| **Target-Based Group Drift** | Deviation indicators measuring drift in the *joint distribution* of each supervised group with the target. |

## 8. Segmented Drift Detection

So far the notebook has compared two flat populations.
APL's `DriftDetector` also supports **segmented detection**: fitting one independent
drift model per segment and returning all results in a single combined table.

**Why segmented detection?**

- Different segments (countries, product lines, regions, …) may have very different
  baseline distributions. A pooled comparison would conflate segment-level patterns.
- Segmented detection lets APL isolate which segments drifted and which did not.
- The same `fit_detect()` API is used — the only difference is the `segment_column_name`
  constructor argument.

**Dataset:** the same census table, restricted to three countries: United States, Mexico,
and Holand-Netherlands. The reference is the male population in those countries; the new
dataset is the female population.

The dataset contains no male respondents from Holand-Netherlands, so that segment will have
an empty reference dataset and is expected to fail. This is intentional — it lets us
demonstrate how APL handles partial failures without aborting the whole operation.

In [ ]:
SEGMENT_COLUMN = "native-country"
SELECTED_COUNTRIES = ["United-States", "Mexico", "Holand-Netherlands"]

country_filter = "(" + ", ".join(f"'{c}'" for c in SELECTED_COUNTRIES) + ")"

hana_ref_seg = (
    hana_census.filter(f'"sex" = \'Male\' AND "{SEGMENT_COLUMN}" IN {country_filter}')
    .sort(SEGMENT_COLUMN)
    .select(FEATURES)  # FEATURES already includes native-country
)

hana_new_seg = (
    hana_census.filter(f'"sex" = \'Female\' AND "{SEGMENT_COLUMN}" IN {country_filter}')
    .sort(SEGMENT_COLUMN)
    .select(FEATURES)
)

print(f"Reference rows: {hana_ref_seg.count()}")
print(f"New rows      : {hana_new_seg.count()}")

In [ ]:
detector_seg = DriftDetector(
    segment_column_name=SEGMENT_COLUMN,
    max_tasks=0,  # use all available HANA threads for parallel segment processing
)

results_seg = detector_seg.fit_detect(
    reference_data=hana_ref_seg,
    new_data=hana_new_seg,
)

print("Segmented drift detection complete.")
print("Deviation indicators per segment:")
results_seg.collect()

The result table includes a `Segment` column. Rows are sorted by segment then by deviation
indicator, so you can immediately see which variables drifted the most within each segment.

### Generating a Report for a Specific Segment

In a segmented model each segment has its own independent drift analysis. The report
is therefore **per-segment**: you must specify which segment to visualize via
`build_report(segment_name=...)`.

In [ ]:
REPORT_SEGMENT = "United-States"

# Build the report data for the chosen segment
detector_seg.build_report(segment_name=REPORT_SEGMENT)

# Display inline
detector_seg.generate_notebook_iframe_report()

### Troubleshooting

Detection may succeed for some segments while failing for others. The overall `fit_detect()`
call does **not** raise an exception in that case — the successful segments still return
results.

**Automatic warning**

When at least one segment fails, APL automatically emits a `WARNING` log message listing the
first 10 failed segments and the corresponding error. You can see this in the `fit_detect()`
output above. For a small number of failures this is usually sufficient to diagnose the problem.

**Checking task status per segment**

When there are more than 10 failures, the warning does not contain the full list. Use
`get_summary()` to retrieve all failed segments programmatically.
Filter on `AplTaskStatus` to get a quick overview of which segments succeeded and which failed:

In [ ]:
df_status = (
    detector_seg.get_summary()
    .filter("\"KEY\" IN ('AplTaskStatus')")
    .select("OID", "VALUE")
    .collect()
)
df_status.columns = [SEGMENT_COLUMN, "Task Status"]
df_status

For any segment that failed, `get_fit_operation_log()` gives the full APL log messages for
that segment.
Filter to `LEVEL = 0` (top-level messages) and the segment's `OID` to surface the root cause:

In [ ]:
df_log = (
    detector_seg.get_fit_operation_log()
    .filter("LEVEL = 0 and OID = 'Holand-Netherlands'")
    .select("OID", "MESSAGE")
    .collect()
)
df_log.columns = [SEGMENT_COLUMN, "Log Text"]
df_log

## 9. Two-Phase Workflow

In production you typically want to:

1. **Fit** the drift model on the reference dataset once (or on a schedule)
2. **Save** it to a persistent HANA table
3. **Load** it in any later session and detect drift on incoming data — without refitting

This two-phase approach is more efficient than `fit_detect()` when:

- The reference dataset is large and refitting is expensive
- You receive multiple batches of new data and want to compare each against the same baseline
- You need a strict audit trail of which reference model was used for a given detection run

`fit()` stores the internal model binary inside SAP HANA, making it available for persistence.
`detect()` applies it to new data and returns deviation indicators just like `fit_detect()`.
`ModelStorage` provides a central registry to save, list, load, and delete models.

In [ ]:
# Phase 1 — fit the reference model
detector_fit = DriftDetector()
detector_fit.fit(reference_data=hana_ref)
print("Reference model fitted.")

In [ ]:
MODEL_NAME = "Census Drift Model"

model_storage = ModelStorage(
    connection_context=conn,
    schema="MODEL_STORAGE",
)

detector_fit.name = MODEL_NAME

# if_exists='replace' overwrites any existing model with the same name and version
model_storage.save_model(model=detector_fit, if_exists="replace")
print(f'Model "{MODEL_NAME}" saved successfully.')

# Verify it appears in the registry
model_storage.list_models(name=MODEL_NAME)

In [ ]:
# Phase 2 — load the model in a later session and detect drift on a new batch
detector_loaded = model_storage.load_model(name=MODEL_NAME)

results_loaded = detector_loaded.detect(
    new_data=hana_new,
    build_report=True,
)

print("Drift detection with the reloaded model:")
results_loaded.collect()

In [ ]:
# Clean up: remove the saved model from the registry
model_storage.delete_model(name=MODEL_NAME, version=1)
print(f'Model "{MODEL_NAME}" removed from storage.')

# Verify it is gone
remaining = model_storage.list_models(name=MODEL_NAME)
if remaining.empty:
    print("No models found — cleanup complete.")